<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/media/banners/banner_googleadspulse_powerbi_guide.png" width="100%"/>
</div>

## 📖 Préambule

### À qui s'adresse ce guide ?

Ce notebook est ton **support de référence** pour construire le tableau de bord *GoogleAdsPulse* dans Power BI Desktop. C'est un dashboard de **paid media analytics** qui agrège la performance Google Ads de 5 comptes clients.

### Comment lire ce guide

| Symbole | Ce qu'il indique |
|---|---|
| 🎯 | Ce que tu sauras faire à la fin de la section |
| 📘 | L'intuition métier ou technique avant de coder |
| 🔧 | Les clics, le code DAX, les paramètres exacts |
| 🎓 | Une méthode opérationnelle pour construire un visuel précis |
| ✅ | Comment vérifier que ton travail est correct |
| ⚠️ | L'erreur courante à éviter |

### Le contexte métier

**GoogleAdsPulse** est un tableau de bord d'agence média qui consolide la performance de 5 comptes Google Ads sur 24 mois. Quatre familles de KPI structurent l'analyse :

1. **Volume** — Impressions, Clicks, Conversions (ce que la campagne génère)
2. **Efficacité** — CTR, CPC, CPM, Conversion Rate (à quel coût et quel taux)
3. **Rentabilité** — Spend, Conversions Value, **ROAS** (retour sur investissement publicitaire)
4. **Variations MoM** — chaque KPI vs mois précédent (couleur sémantique : ↑ ou ↓ selon la nature)

Le dashboard répond à 5 questions :

| Page | Question |
|---|---|
| 1 — Overview | Quelle est la performance globale des 5 comptes sur 24 mois ? |
| 2 — Campaigns | Quelles campagnes sont rentables (ROAS > 3×) et lesquelles à couper ? |
| 3 — Keywords | Quels mots-clés performent dans le temps (cohort retention) ? |
| 4 — Conversions | Comment se répartissent les 60K conversions par type ? |
| 5 — Breakdown | Quel device, quel jour/heure, quel pays, quel type de campagne dépense le plus ? |

---
# I — Préparer les fondations

## 1.1 Comprendre les sources de données

### 📘 Concept clé — sources brutes vs sources agrégées

Le projet utilise **deux familles de sources** :

**Sources brutes Google Ads** (4 tables) — exports natifs de l'API Google Ads :
- `accounts` — 5 comptes clients (TechShop, AfriHotels, BankAfrica, MedSupply, EdTech)
- `campaigns` — 25 campagnes actives (Search, Display, Performance Max, Shopping)
- `keywords` — 372 mots-clés avec Quality Score
- `ads` — créations publicitaires liées aux campagnes

**Performance brute quotidienne** (1 table) :
- `performance_quotidienne` — 1 ligne = 1 campagne × 1 jour × 1 device avec impressions, clicks, conversions, spend

**Sources agrégées** sortant du notebook Python (7 tables) :
- `gads_kpi_mensuel` — KPIs mensuels pré-calculés (snapshot par compte × mois)
- `gads_campaigns_rank` — classement des campagnes par ROAS avec quartile et verdict
- `gads_cohort_keywords` — table de rétention cohorte des mots-clés (1 ligne = 1 keyword × M0…M12)
- `gads_cohort_long` — version longue de la cohort (pour heatmap natif Power BI)
- `gads_jour_heure` — 7 jours × 8 tranches horaires (pour heatmap)
- `gads_anomalies` — alertes spend / CPC / CTR détectées
- `gads_benchmark` — benchmarks sectoriels par campagne
- `gads_rolling` — moyennes glissantes 7j / 28j

### ⚠️ Piège — quelle table utiliser pour quel KPI ?

- **KPIs Volume / Efficacité / Rentabilité** ⇒ `performance_quotidienne` (somme propre)
- **Variations MoM** ⇒ `gads_kpi_mensuel` (déjà décalé d'un mois en amont)
- **Top/Bottom ROAS, quartile** ⇒ `gads_campaigns_rank`
- **Heatmap Jour × Heure** ⇒ `gads_jour_heure`
- **Cohort retention** ⇒ `gads_cohort_keywords` (format wide M0-M12) ou `gads_cohort_long`

## 1.2 Importer les CSV

### 📘 Concept clé — pourquoi GitHub raw plutôt que des fichiers locaux ?

Charger depuis une URL `raw.githubusercontent.com` te donne deux superpouvoirs :
1. **Reproductibilité** : tous les apprenants ont exactement la même donnée, à l'octet près.
2. **Mise à jour facile** : si on corrige une coquille dans le CSV, un simple *Actualiser* suffit, aucune ré-installation.

### Les URLs à utiliser (12 CSV)

```
# Sources brutes
https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/googleadspulse_analytics/data/accounts.csv
https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/googleadspulse_analytics/data/campaigns.csv
https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/googleadspulse_analytics/data/keywords.csv
https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/googleadspulse_analytics/data/ads.csv
https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/googleadspulse_analytics/data/performance_quotidienne.csv

# Sources agrégées (sortie notebook Python)
https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/googleadspulse_analytics/corrige/outputs/gads_kpi_mensuel.csv
https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/googleadspulse_analytics/corrige/outputs/gads_campaigns_rank.csv
https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/googleadspulse_analytics/corrige/outputs/gads_cohort_keywords.csv
https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/googleadspulse_analytics/corrige/outputs/gads_cohort_long.csv
https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/googleadspulse_analytics/corrige/outputs/gads_jour_heure.csv
https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/googleadspulse_analytics/corrige/outputs/gads_anomalies.csv
https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/googleadspulse_analytics/corrige/outputs/gads_benchmark.csv
https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/googleadspulse_analytics/corrige/outputs/gads_rolling.csv
```



<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/googleadspulse/powerbi/tuto/01.png" style="width:100%; max-width:1000px; height:auto;"/>
</div>

## 1.3 Désactiver l'Auto Date/Time

**Fichier → Options → Chargement des données → décocher Date/heure automatique**.

Sans ça, Power BI crée une LocalDateTable cachée pour chaque colonne de date — sur ce projet (`reservations[date_arrivee]`, `reservations[date_depart]`, `paiements[date_paiement]`, `services[date_service]`...), c'est 4-5 tables fantômes en moins.


<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/ecommerce_analytics/powerbi/tuto/02_options_auto_datetime.png" style="width:100%; max-width:1000px; height:auto;"/>
</div>

Critique sur ce projet : `performance_quotidienne[date]`, `campaigns[start_date]`, `gads_kpi_mensuel[mois]`, `gads_anomalies[date]` — sans désactivation, 4-5 LocalDateTables fantômes seraient créées.

---
# II — Modéliser les données

## 2.1 Schéma en étoile

### 📘 Concept clé — `performance_quotidienne` est la table pivot

C'est une table de faits massive (~50 000 lignes : 25 campagnes × 730 jours × 3 devices). Toutes les dimensions s'y branchent : `accounts`, `campaigns`, `keywords` (via campaign_id), et le `dim_calendrier`.

### Diagramme du modèle

```
                       +---------------------+
                       |    dim_calendrier    |
                       +---------+-----------+
                                 | 1
                                 | N
    +-----------+   1   N   +----+--------------------+   N   1   +-----------+
    |  accounts +-----------+ performance_quotidienne +-----------+ campaigns |
    +-----------+           +---------+----+----------+           +-----+-----+
                                      |    | N                          | 1
                                  N   |    +-------+                  N |
                                      | 1          | 1            +-----+-----+
                              +-------+----+    +--+------+       | keywords  |
                              |    ads     |    |  ads    |       +-----------+
                              +------------+    +---------+              |
                                                                         | N
                                                                         | 1
                                                              +----------+----------+
                                                              | gads_cohort_keywords|
                                                              +---------------------+

Tables de faits dérivées (autonomes, branchées soit sur calendrier soit campaigns) :
  - gads_kpi_mensuel    (mois)        ← dim_calendrier
  - gads_campaigns_rank (campaign_id) ← campaigns
  - gads_cohort_long    (cohorte mois)
  - gads_jour_heure     (jour, tranche)
  - gads_anomalies      (date)        ← dim_calendrier
  - gads_benchmark      (campaign_id) ← campaigns
  - gads_rolling        (date)        ← dim_calendrier
```

## 2.2 Créer la table `dim_calendrier`

**Modélisation → Nouvelle table** :

```dax
dim_calendrier = 
ADDCOLUMNS(
    CALENDAR(DATE(2023,1,1), DATE(2024,12,31)),
    "Annee",         YEAR([Date]),
    "Mois_Num",      MONTH([Date]),
    "Mois_Nom",      FORMAT([Date], "mmm", "fr-FR"),
    "Annee_Mois",    FORMAT([Date], "yyyy-MM"),
    "Trimestre",     "T" & QUARTER([Date]),
    "Jour_Semaine",  FORMAT([Date], "dddd", "fr-FR"),
    "Jour_Sem_Ordre",WEEKDAY([Date], 2)
)
```

## 2.3 Marquer le calendrier comme table de dates

Vue Données → `dim_calendrier` → **Outils de table → Marquer comme table de dates → colonne Date**.

<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/googleadspulse/powerbi/tuto/04.png" style="width:100%; max-width:1000px; height:auto;"/>
</div>


## 2.4 Établir les relations

| # | De (1) | Clé | Vers (N) | Clé | Direction |
|---|---|---|---|---|---|
| 1 | `dim_calendrier` | `Date` | `performance_quotidienne` | `date` | Single |
| 2 | `dim_calendrier` | `Date` | `gads_kpi_mensuel` | `mois` | Single |
| 3 | `dim_calendrier` | `Date` | `gads_anomalies` | `date` | Single |
| 4 | `dim_calendrier` | `Date` | `gads_rolling` | `date` | Single |
| 5 | `accounts` | `account_id` | `performance_quotidienne` | `account_id` | Single |
| 6 | `accounts` | `account_id` | `campaigns` | `account_id` | Single |
| 7 | `campaigns` | `campaign_id` | `performance_quotidienne` | `campaign_id` | Single |
| 8 | `campaigns` | `campaign_id` | `keywords` | `campaign_id` | Single |
| 9 | `campaigns` | `campaign_id` | `ads` | `campaign_id` | Single |
| 10 | `campaigns` | `campaign_id` | `gads_campaigns_rank` | `campaign_id` | Single |
| 11 | `campaigns` | `campaign_id` | `gads_benchmark` | `campaign_id` | Single |




<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/googleadspulse/powerbi/tuto/03.png" style="width:100%; max-width:1000px; height:auto;"/>
</div>

---
# III — Créer la table `_Mesures`

**Accueil → Entrer des données →** 1 colonne, 1 ligne vide → nommer `_Mesures` → **Charger**.

Après avoir créé ta première mesure et l'avoir glissée dans `_Mesures`, supprime la colonne fictive.

---
# IV — Construire les 82 mesures DAX

### Vue d'ensemble des 11 dossiers

| Dossier | Mesures | Rôle |
|---|---|---|
| `0. Sous-titres` | 5 | 1 sous-titre dynamique par page |
| `1. Volume` | 3 | Total Impressions, Clicks, Conversions |
| `2. Efficacité` | 5 | CTR, CPC, CPM, Cost per Conversion, Conversion Rate |
| `3. Rentabilité` | 3 | Total Spend, Conversions Value, ROAS |
| `4. Période précédente` | 10 | Valeur de chaque KPI au mois précédent (M-1) |
| `5. Deltas MoM` | 10 | Variation % de chaque KPI vs M-1 |
| `6. Conversions par type` | 5 | Purchase, Lead, Signup, Demo + Purchase Value |
| `7. Contextuelles` | 11 | ROAS Verdict, Spend YTD, YoY%, % Spend Account, QS Moyen, Trimestre Pic, etc. |
| `8. Keywords KPI` | 4 | Nb Keywords actifs, Keywords à pauser, Top CTR keyword |
| `9. Devices` | 3 | % Clicks Desktop / Mobile / Tablet |
| `10. HTML Visuels` | 1 | Heatmap Jour × Heure HTML |
| `_Helpers` | 19 | Mesures couleur dynamiques pour mise en forme conditionnelle |

## 4.1 Dossiers `1-3` — Volume / Efficacité / Rentabilité (11 mesures)

Toutes ces mesures s'appuient sur `performance_quotidienne`.

```dax
Total Impressions = SUM(performance_quotidienne[impressions])
Total Clicks = SUM(performance_quotidienne[clicks])
Total Conversions = SUM(performance_quotidienne[conversions])

CTR = DIVIDE([Total Clicks], [Total Impressions])
CPC = DIVIDE([Total Spend], [Total Clicks])
CPM = DIVIDE([Total Spend] * 1000, [Total Impressions])
Cost per Conversion = DIVIDE([Total Spend], [Total Conversions])
Conversion Rate = DIVIDE([Total Conversions], [Total Clicks])

Total Spend = SUM(performance_quotidienne[cost_eur])
Conversions Value = SUM(performance_quotidienne[conversions_value])
ROAS = DIVIDE([Conversions Value], [Total Spend])
```

### 📘 Sémantique des seuils ROAS

| ROAS | Verdict | Couleur |
|---|---|---|
| ≥ 5× | Très rentable ⭐ | Vert leader |
| ≥ 3× | Rentable ✅ | Vert |
| ≥ 1× | Limite ⚠️ | Orange |
| < 1× | Non rentable ❌ | Rouge |

Le ROAS = chiffre d'affaires généré par 1€ dépensé. **Seuil de rentabilité = 3×** (couvre les coûts média + marge agence + marge client).

## 4.2 Dossiers `4-5` — Périodes précédentes & Deltas MoM (20 mesures)

### Pattern type — KPI au mois précédent

```dax
Impressions Prev = 
CALCULATE(
    [Total Impressions],
    DATEADD(dim_calendrier[Date], -1, MONTH)
)
```
*Variante avec `PREVIOUSMONTH` : `CALCULATE([Total Impressions], PREVIOUSMONTH(dim_calendrier[Date]))`. `DATEADD` est plus flexible si on veut comparer à N-1 mois ou N-1 année.*

À reproduire pour les **10 KPIs** : Impressions, Clicks, Conversions, CTR, CPC, CPM, Cost per Conversion, Conversion Rate, Total Spend, Conversions Value.

### Pattern type — Delta MoM

```dax
Impressions Delta % = 
DIVIDE([Total Impressions] - [Impressions Prev], [Impressions Prev])
```

### ⚠️ Sémantique inversée pour CPC, CPM, Cost per Conversion

Pour la plupart des KPIs : **↑ = vert (bon), ↓ = rouge (mauvais)** — c'est la logique « UP IS GOOD ».

Mais pour **CPC, CPM, Cost per Conversion** : **↓ = vert (bon)** — c'est la logique « DOWN IS GOOD » (on veut payer moins cher le clic / l'acquisition).

Cette sémantique est encodée dans les mesures `_Helpers/Couleur ...` du dossier `_Helpers` (cf. §4.7).

## 4.3 Dossier `6. Conversions par type` (5 mesures)

```dax
Purchase Conversions = CALCULATE([Total Conversions], performance_quotidienne[conversion_type] = "Purchase")
Lead Conversions = CALCULATE([Total Conversions], performance_quotidienne[conversion_type] IN {"Lead", "Form Submit", "Phone Call"})
Signup Conversions = CALCULATE([Total Conversions], performance_quotidienne[conversion_type] = "Signup")
Demo Conversions = CALCULATE([Total Conversions], performance_quotidienne[conversion_type] = "Demo Request")

Purchase Value = CALCULATE([Conversions Value], performance_quotidienne[conversion_type] = "Purchase")
```

### 📘 Pourquoi grouper Lead + Form Submit + Phone Call ?

Pour les clients B2B (BankAfrica, MedSupply), un « lead qualifié » se manifeste sous trois formes : remplissage de formulaire, appel téléphonique, ou inscription explicite à une newsletter. Les comptabiliser séparément serait artificiel — c'est la même intention business.

## 4.4 Dossier `7. Contextuelles` (11 mesures)

Mesures spécifiques aux pages Campaigns / Keywords / Breakdown :

```dax
ROAS Verdict = 
VAR _r = [ROAS]
RETURN SWITCH(TRUE(),
    _r >= 5, "⭐ Tres rentable",
    _r >= 3, "✅ Rentable",
    _r >= 1, "⚠️ Limite",
    "❌ Non rentable"
)

ROAS Verdict Icone = 
VAR _r = [ROAS]
RETURN SWITCH(TRUE(), _r >= 5, "⭐", _r >= 3, "✅", _r >= 1, "⚠️", "❌")

Spend 7d MA = 
AVERAGEX(
    DATESINPERIOD(dim_calendrier[Date], MAX(dim_calendrier[Date]), -7, DAY),
    [Total Spend]
)

Spend YTD = TOTALYTD([Total Spend], dim_calendrier[Date])
Spend YoY % = DIVIDE([Total Spend] - CALCULATE([Total Spend], SAMEPERIODLASTYEAR(dim_calendrier[Date])), CALCULATE([Total Spend], SAMEPERIODLASTYEAR(dim_calendrier[Date])))

Nb Campagnes Actives = 
CALCULATE(
    DISTINCTCOUNT(performance_quotidienne[campaign_id]),
    performance_quotidienne[clicks] > 0
)

QS Moyen = AVERAGE(keywords[quality_score])
Cost per Purchase = DIVIDE([Total Spend], [Purchase Conversions])
Qrt = SELECTEDVALUE(gads_campaigns_rank[quartile_roas])
```

## 4.5 Dossier `8. Keywords KPI` (4 mesures)

```dax
Nb Keywords Actifs = CALCULATE(COUNTROWS(keywords), keywords[status] = "Enabled")

Keywords A Pauser = 
CALCULATE(
    COUNTROWS(keywords),
    keywords[quality_score] <= 3,
    keywords[total_spend] > 100,
    keywords[status] = "Enabled"
)

Top CTR Keyword Nom = 
VAR _kw = TOPN(1,
    FILTER(keywords, keywords[total_impressions] >= 1000),
    DIVIDE(keywords[total_clicks], keywords[total_impressions]),
    DESC
)
RETURN MAXX(_kw, keywords[keyword_text]) & " (" & MAXX(_kw, keywords[match_type]) & ")"

Top CTR Keyword Valeur = 
VAR _kw = TOPN(1,
    FILTER(keywords, keywords[total_impressions] >= 1000),
    DIVIDE(keywords[total_clicks], keywords[total_impressions]),
    DESC
)
RETURN MAXX(_kw, DIVIDE(keywords[total_clicks], keywords[total_impressions]))
```

### 📘 Critère « Keywords à pauser »

Trois conditions cumulatives :
1. **Quality Score ≤ 3** (faible pertinence Google Ads)
2. **Spend > 100 €** (seulement les vrais consommateurs de budget — exclus les keywords qui ne dépensent rien)
3. **Status = Enabled** (déjà actifs, sinon ils sont déjà pausés)

C'est la liste prioritaire d'optimisation pour le manager de compte.

## 4.6 Dossier `9. Devices` (3 mesures) + `10. HTML Visuels` (1 mesure)

```dax
% Clicks Desktop = DIVIDE(CALCULATE([Total Clicks], performance_quotidienne[device] = "Desktop"), [Total Clicks])
% Clicks Mobile = DIVIDE(CALCULATE([Total Clicks], performance_quotidienne[device] = "Mobile"), [Total Clicks])
% Clicks Tablet = DIVIDE(CALCULATE([Total Clicks], performance_quotidienne[device] = "Tablet"), [Total Clicks])
```

### Heatmap Jour × Heure HTML

```dax
Heatmap Jour Heure HTML = 
// Mesure HTML qui rend une grille 7 jours × 8 tranches horaires
// Pattern CONCATENATEX + style inline
// Couleurs 6 paliers : pâle → rouge orange selon le spend
// Fond TRANSPARENT (s'adapte au fond de la page)
// À utiliser avec le visuel marketplace HTML Content (Daniel Marsh-Patrick)
```

## 4.7 Dossier `_Helpers` — Mesures couleur dynamiques (19 mesures)

### 📘 Concept clé — couleur sémantique différenciée

Sur GoogleAdsPulse, la couleur du delta n'est PAS toujours « vert si positif ». Elle dépend de la **nature du KPI** :

**UP IS GOOD** (vert si delta > 0) : Impressions, Clicks, Conversions, CTR, Conversion Rate, Conversions Value, Total Spend

**DOWN IS GOOD** (vert si delta < 0) : CPC, CPM, Cost per Conversion

### Pattern UP IS GOOD

```dax
Couleur Conversions = 
VAR _d = [Conversions Delta %]
RETURN SWITCH(TRUE(),
    _d >= 0, "#10B981",
    "#EF4444"
)
```

### Pattern DOWN IS GOOD

```dax
Couleur CPC = 
VAR _d = [CPC Delta %]
RETURN SWITCH(TRUE(),
    _d <= 0, "#10B981",
    "#EF4444"
)
```

### Application sur les cartes Page 1

Sur chaque carte KPI (Page Overview), la valeur du delta s'affiche avec :
- Format → **Couleur de la police** → fx → Valeur du champ → mesure couleur correspondante
- Exemple : carte CPC → Couleur de la police du delta → `Couleur CPC` (vert si CPC baisse, rouge sinon)

## 4.8 Dossier `0. Sous-titres` (5 mesures)

Une mesure par page, branchée sur une **Carte** ou une **Zone de texte** sous le titre.

```dax
Sous Titre Page 01 = 
VAR _compte = SELECTEDVALUE(accounts[account_name], "5 comptes")
VAR _annee  = SELECTEDVALUE(dim_calendrier[Annee], "24 mois")
RETURN "Vue globale des " & _compte & " · Performance " & _annee

Sous Titre Page 02 = [Nb Campagnes Actives] & " campagnes actives · Performance par ROAS"
Sous Titre Page 03 = [Nb Keywords Actifs] & " mots-cles · Performance et retention dans le temps"
Sous Titre Page 04 = FORMAT([Total Conversions], "#,0") & " conversions · Repartition par type et cout unitaire"
Sous Titre Page 05 = "Analyse par device, horaire, geographie et type de campagne"
```

---
# V — Design system

## 5.1 Charte GoogleAdsPulse — Light + Violet

| Rôle | Hex | Usage |
|---|---|---|
| Primaire (violet) | `#7C3AED` | Logo, navbar item actif, accents purple |
| Secondaire (teal) | `#10B981` | KPI rentables, ROAS ≥ 3× |
| Warning (orange) | `#F59E0B` | KPI limite, ROAS 1-3× |
| Danger (rouge) | `#EF4444` | Non rentable, ROAS < 1× |
| Info (bleu) | `#3B82F6` | Mobile devices |
| Texte principal | `#0F172A` | Hero numbers, titres |
| Texte secondaire | `#64748B` | Labels, axes, sous-titres |
| Fond page | `#F8FAFC` | Arrière-plan |
| Card background | `#FFFFFF` | Cards |

## 5.2 Mockup PowerPoint → fonds PNG d'arrière-plan (A télécharger sur le plateforme)

### 📘 Concept clé

Power BI gère mal les arrière-plans complexes (cards à ombre, navbar en haut). La méthode pro : dessiner dans **PowerPoint** (mockup vierge sans données), exporter en **PNG haute résolution** (1280×720), importer comme **arrière-plan de page**, poser les visuels Power BI **par-dessus**.

### Renommage final

```
bg-01-overview.png
bg-02-campaigns.png
bg-03-keywords.png
bg-04-conversions.png
bg-05-breakdown.png
```

### 🔧 Méthode 1 — Export PNG depuis PowerPoint à 150 DPI

1. **Win + R** → `regedit` → **Entrée**
2. Aller dans `HKEY_CURRENT_USER\Software\Microsoft\Office\16.0\PowerPoint\Options`
3. Clic droit → **Nouveau** → **Valeur DWORD (32 bits)** → Nom : `ExportBitmapResolution`, Valeur : `150` (décimal)
4. Redémarrer PowerPoint, **Fichier → Enregistrer sous → PNG → Toutes les diapositives**

### 🔧 Méthode 2 — CloudConvert

[cloudconvert.com/pptx-to-png](https://cloudconvert.com/pptx-to-png) → upload `mockup_googleadspulse_blank.pptx` → 150 DPI → 1280×720.

### 🔧 Application dans Power BI

1. Sélectionner la page → **Format de la page** (icône pinceau)
2. **Arrière-plan de la page** → **Ajouter une image** → choisir le PNG
3. **Ajustement** → **Adapter** · **Transparence** → **0 %**


<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/googleadspulse/powerbi/tuto/02.png" style="width:100%; max-width:1000px; height:auto;"/>
</div>

---
# VI — Construire les 5 pages

Cette partie détaille **chaque visuel** avec sa configuration exacte (type, axes, couleurs, étiquettes) et les **méthodes Power BI** non-triviales nécessaires pour le rendu final.

## 6.1 Page 1 — Overview

> *« Quelle est la performance globale des 5 comptes sur 24 mois ? »*

**1. Slicers globaux (2 dates)**

- Type : 2 Slicers Date (« 01/01/2023 » et « 31/12/2024 »)
- Champ : `dim_calendrier[Date]` mode "Entre" pour filtre période
- Format : compact, icône calendrier, fond blanc, bordure grise

**2-11. 10 KPI cards avec delta MoM coloré**

Disposition : 2 lignes × 5 cards. Pour chaque card :

- Type : Carte avec icône en haut-gauche
- Icône : Unicode (👁 Impressions, 🖱 Clicks, 🎯 Conversions, 📈 CTR, 💰 CPC, 💸 Spend, 📊 CPM, ⚠️ Cost/Conv, 📈 Conv. Rate, 📈 Conv. Value)
- Label haut : nom du KPI Segoe UI 11pt gris
- Valeur : Segoe UI Bold 28pt noir formatée (`56,02M`, `1,49M`, `59,7K`, `2,66%`, `€0,42`...)
- Delta : à droite Segoe UI 12pt avec couleur dynamique (`+6,8 % ↑` vert ou `-0,1 % ↓` rouge)

> 🎓 **METHODE — Carte KPI avec valeur + delta coloré sur la même ligne**
>
> Power BI Carte simple ne permet qu'une seule valeur. Solution : utiliser **Carte avec plusieurs lignes** ou **Visuel personnalisé KPI Card** (marketplace). Alternative simple :
>
> 1. Insérer 1 Carte pour la valeur principale (`Total Impressions` formatée "56,02M")
> 2. Insérer 1 Carte pour le delta (`Impressions Delta %` formatée "+0.0%;-0.0%" + flèche Unicode)
> 3. Format → Couleur de la police du delta → fx → Valeur du champ → `Couleur Impressions`
> 4. Aligner les 2 cartes côte à côte sans espace, puis grouper (Format → Grouper)

**12. Combo « Impressions vs Clicks »**

- Type : Histogramme et courbe groupés (combo chart)
- Axe X : `dim_calendrier[Mois_Nom]`
- Axe Y1 (barres bleu clair `#60A5FA`) : `Total Impressions`, label « Impressions (M) »
- Axe Y2 (ligne bleu marine `#1E3A8A`) : `Total Clicks`, label « Total Clicks »
- Légende : en haut à droite (« Total Impressions », « Total Clicks »)

> 🎓 **METHODE — Combo chart avec axe Y double**
>
> Le visuel **Histogramme et courbe empilés** (ou **groupés**) supporte nativement 2 axes Y. Glisser la mesure 1 dans **Valeurs de colonne**, la mesure 2 dans **Valeurs de ligne**. Format → **Axe Y** → Plage de valeurs → Activer **Axe secondaire**.

**13. Ligne « CPC evolution (€) »**

- Type : Ligne avec marqueurs et étiquettes
- Axe X : `dim_calendrier[Mois_Nom]`
- Axe Y : `CPC`
- Couleur ligne : orange `#F59E0B`, épaisseur 2px
- Marqueurs : ronds orange taille 6
- Étiquettes de données : valeur `€0,XX` au-dessus de chaque point

**14. Aire « Spend amount by Date »**

- Type : Aire (zone)
- Axe X : `dim_calendrier[Date]` (granularité jour ou mois)
- Axe Y : `Total Spend`
- Couleur de remplissage : orange clair `#FED7AA` 60% opacité
- Couleur de la ligne : orange `#F97316`
- Pas d'étiquettes (la courbe est lisible seule)

**15. Ligne « Conversions mensuelles (K) »**

- Type : Ligne avec marqueurs et étiquettes
- Axe X : `dim_calendrier[Mois_Nom]`
- Axe Y : `Total Conversions`
- Couleur ligne : vert `#10B981`, épaisseur 2px
- Marqueurs : ronds verts taille 6
- Étiquettes : valeur en K avec couleur conditionnelle (vert si > moyenne, gris sinon)



<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/googleadspulse/powerbi/tuto/01_page_overview.png" style="width:100%; max-width:1000px; height:auto;"/>
</div>

## 6.2 Page 2 — Campaigns

> *« Quelles campagnes sont rentables (ROAS > 3×) et lesquelles à couper ? »*

**1. Top 5 campaigns by ROAS**

- Type : Barres horizontales
- Axe Y : `campaigns[campaign_name]`
- Axe X : `ROAS`
- Filtre : Top N = 5 par `ROAS` DESC
- Couleur barres : vert solide `#10B981` (toutes rentables)
- Étiquettes : valeur formatée `XX,XX×` à droite des barres
- Tri : descendant
- Titre : « ⭐ Top 5 campaigns by ROAS »

**2. Bottom 5 campaigns by ROAS**

- Type : Barres horizontales
- Axe Y : `campaigns[campaign_name]`
- Axe X : `ROAS`
- Filtre : Bottom N = 5 par `ROAS` ASC
- Couleur barres : rouge `#EF4444` (toutes en alerte)
- Étiquettes : valeur formatée `X,XX×` à droite
- Tri : ascendant
- Titre : « ⭐ Bottom 5 campaigns by ROAS »

> 🎓 **METHODE — Filtre Top N sur un visuel**
>
> Volet **Filtres** → glisser `campaigns[campaign_name]` dans « Filtres sur ce visuel » → **Type de filtre** : Top N → **Afficher les éléments** : 5, **Par valeur** : `ROAS`. Pour le Bottom 5, sélectionner **Bas** au lieu de **Haut**.

**3. Tableau « Toutes les campagnes — Classement et verdict »**

- Type : Table
- Colonnes : `campaigns[campaign_name]`, `campaigns[type]`, `Total Spend`, `Total Clicks`, `Total Conversions`, `CTR`, `Cost per Conversion`, `ROAS`, `ROAS Verdict`, `Qrt`
- En-tête : fond gris pâle `#F1F5F9`, texte gris foncé
- Tri : descendant sur `ROAS`
- Mise en forme conditionnelle sur la colonne `ROAS Verdict` :
  - ⭐ Très rentable → texte vert `#10B981`
  - ✅ Rentable → texte vert `#10B981`
  - ⚠️ Limite → texte orange `#F59E0B`
  - ❌ Non rentable → texte rouge `#EF4444`

> 🎓 **METHODE — Mise en forme conditionnelle texte par règle ROAS Verdict**
>
> Sur la colonne `ROAS Verdict` du tableau : Format → **Cellules** → **Couleur de la police** → Mettre en forme par **Règles** → ajouter 4 règles selon la valeur exacte du verdict ("⭐ Tres rentable" → vert, etc.). Alternative : créer une mesure `Couleur ROAS Verdict` qui retourne un hex et utiliser **Valeur du champ** pour appliquer la couleur dynamiquement.



<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/googleadspulse/powerbi/tuto/02_page_campaigns.png" style="width:100%; max-width:1000px; height:auto;"/>
</div>

## 6.3 Page 3 — Keywords

> *« Quels mots-clés performent dans le temps (cohort retention) ? »*

**1-3. 3 KPI cards Keywords**

- KPI 1 — **Quality Score moyen** : icône ✓ vert, valeur `[QS Moyen]` formatée `0,0` (ex: 6,7), couleur valeur vert `#10B981`
- KPI 2 — **Keywords à pauser** : icône ⏸ orange, valeur `[Keywords A Pauser]` (ex: 23), couleur valeur orange `#F59E0B`
- KPI 3 — **Top CTR keyword** : icône ⚡ violet, valeur `[Top CTR Keyword Valeur]` formatée `0,0%` (ex: 17,0%), sous-titre `[Top CTR Keyword Nom]` (ex: « brand hôtel sénégal p (Exact) »), couleur valeur violet `#7C3AED`

**4. Cohort Retention — Keywords par mois du 1er clic**

- Type : Matrice native
- Lignes : `gads_cohort_keywords[cohorte_mois]` (2023-01, 2023-08, 2023-09, 2023-10, etc.)
- Colonnes : `gads_cohort_keywords[mois_relatif]` (M0, M1, M2, ..., M12)
- Valeurs : `gads_cohort_keywords[pct_actifs]` formaté `0,00`
- Mise en forme conditionnelle (par cellule) : gradient vert pâle → vert foncé
- Sous-titre : « Lignes = cohorte d'apparition · Colonnes = mois relatif · Valeur = % de keywords encore actifs »

> 🎓 **METHODE — Cohort retention en matrice native**
>
> 1. La table source `gads_cohort_keywords` doit être au format **wide** (1 ligne par cohorte, colonnes M0-M12). Si elle est en **long**, dépivoter en Power Query.
> 2. Visuel Matrice → Lignes : `cohorte_mois`, Colonnes : `mois_relatif`, Valeurs : `pct_actifs`
> 3. Format → **Éléments de cellule** → activer **Couleur d'arrière-plan** → Avancé → **Mise en forme par : Règles**
> 4. Définir un gradient : valeur 0 = blanc `#FFFFFF`, valeur 100 = vert foncé `#10B981`
> 5. Désactiver les totaux (Format → Totaux → Désactivé)



<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/googleadspulse/powerbi/tuto/03_page_keywords.png" style="width:100%; max-width:1000px; height:auto;"/>
</div>

## 6.4 Page 4 — Conversions

> *« Comment se répartissent les 60K conversions par type ? »*

**1. Donut « Conversions par type »**

- Type : Anneau (Donut)
- Catégorie : `performance_quotidienne[conversion_type]` (Purchase, Lead, Signup, Form Submit, Phone Call, Demo Request)
- Valeur : `Total Conversions`
- Couleurs : Purchase violet `#7C3AED` (63%), Lead vert `#10B981` (17%), Signup bleu `#3B82F6` (7%), Form Submit jaune `#FBBF24` (7%), Phone Call rouge `#EF4444` (4%), Demo Request rose `#F472B6` (3%)
- Étiquettes : pourcentage à l'intérieur du donut
- Trou central : 70%
- Légende : à droite vertical

**2. Barres « Cost per Conversion par type (€) »**

- Type : Barres horizontales
- Axe Y : `performance_quotidienne[conversion_type]`
- Axe X : `Cost per Conversion`
- Couleur barres : violet uniforme `#7C3AED`
- Étiquettes : valeur `€XX,XX` à droite de chaque barre
- Tri : descendant

**3. Barres « Conversion Value par type (K€) »**

- Type : Barres verticales
- Axe X : `performance_quotidienne[conversion_type]`
- Axe Y : `Conversions Value`
- Couleur barres : vert uniforme `#10B981`
- Étiquettes : valeur formatée K€ au-dessus de chaque barre (ex: `€2 925,93K`)
- Tri : descendant

> 🎓 **METHODE — Format "K€" en étiquette de données**
>
> Sur la mesure ou la colonne, appliquer le format custom `"€"#,0,"K"` (la virgule après #,0 divise par 1000 et le K ajoute le suffixe). Pour M : `"€"#,0,,"M"`. Si tu utilises une mesure dédiée : `Conv Value Label = FORMAT([Conversions Value]/1000, "€#,0") & "K"`.

**4. Tableau « Top 8 campagnes par nombre de conversions »**

- Type : Table
- Colonnes : `campaigns[campaign_name]`, `campaigns[type]`, `Total Conversions`, `Conversions Value`
- Filtre : Top 8 par `Total Conversions`
- En-tête : fond `#F1F5F9`, texte gris foncé
- Tri : descendant sur Conversions
- Format colonnes Value : `€XXX K` (FORMAT custom)



<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/googleadspulse/powerbi/tuto/04_page_conversions.png" style="width:100%; max-width:1000px; height:auto;"/>
</div>

## 6.5 Page 5 — Breakdown

> *« Quel device, quel jour/heure, quel pays, quel type de campagne dépense le plus ? »*

**1. Performance par Device**

- Type : 3 cards multi-rows + 3 mini bar charts en bas
- Card 1 (Desktop) : icône 🖥 violet, valeur `[% Clicks Desktop]` (ex: 43%), sous-texte « Spend €265,4K » (= `CALCULATE([Total Spend], performance_quotidienne[device]="Desktop")` formaté en K€)
- Card 2 (Mobile) : icône 📱 bleu, valeur `[% Clicks Mobile]` (52%), sous-texte « Spend €329,9K »
- Card 3 (Tablet) : icône 📲 orange, valeur `[% Clicks Tablet]` (5%), sous-texte « Spend €30,3K »
- Mini barres : 3 barres verticales violettes représentant `[CTR]` par device (ex: Mobile 4,0% / Desktop 4,0% / Tablet 3,9%)

**2. Heatmap Jour × Heure (spend)**

- Type : Matrice native ou **HTML Content** (selon la complexité voulue)
- Lignes : `gads_jour_heure[jour]` (Lun, Mar, Mer, Jeu, Ven, Sam, Dim)
- Colonnes : `gads_jour_heure[tranche_horaire]` (0-3, 3-6, 6-9, 9-12, 12-15, 15-18, 18-21, 21-24)
- Valeurs : `gads_jour_heure[spend_keur]`
- Mise en forme conditionnelle : 6 paliers (pâle → orange foncé → rouge)

> 🎓 **METHODE — Heatmap matricielle en visuel natif**
>
> 1. Visuel Matrice → Lignes : `jour`, Colonnes : `tranche_horaire`, Valeurs : `spend_keur`
> 2. Format → Éléments de cellule → **Couleur d'arrière-plan** → fx → **Mise en forme par : Règles**
> 3. Définir 6 règles :
>    - 0 ≤ valeur < 3 → blanc `#FFFFFF`
>    - 3 ≤ valeur < 7 → orange très clair `#FFEDD5`
>    - 7 ≤ valeur < 12 → orange clair `#FED7AA`
>    - 12 ≤ valeur < 18 → orange `#FB923C`
>    - 18 ≤ valeur < 25 → orange foncé `#F97316`
>    - ≥ 25 → rouge `#EA580C`
> 4. Désactiver les totaux
> 5. Format → Valeurs → Police 11pt selon contraste

**3. Spend par Pays**

- Type : Barres horizontales
- Axe Y : `performance_quotidienne[country]`
- Axe X : `Total Spend`
- Couleur barres : vert dégradé `#10B981` (top 1) → `#86EFAC` (rangs suivants)
- Étiquettes : valeur formatée K€ à droite (`€248,9K`, `€123,8K`...)
- Tri : descendant
- Icône globe vert en haut à droite (zone de texte avec emoji 🌍)

**4. Répartition Spend par Campaign Type (par année)**

- Type : Barres empilées 100%
- Axe X : `dim_calendrier[Annee]` (2023, 2024)
- Axe Y empilé : `Total Spend`
- Légende : `campaigns[type]` (Display bleu, Performance Max violet, Search vert, Shopping orange, Video rouge)
- Étiquettes : pas d'étiquettes (la lisibilité du stack suffit)
- Position légende : en bas



<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/googleadspulse/powerbi/tuto/05_page_breakdown.png" style="width:100%; max-width:1000px; height:auto;"/>
</div>

---
# VII — Slicers, navigation, finitions

**Navigation horizontale en haut de chaque page** : 5 boutons « Overview · Campaigns · Keywords · Conversions · Breakdown ». Le bouton actif a un soulignement violet `#7C3AED` 2px et le texte en violet ; les autres en gris `#64748B`.

> 🎓 **METHODE — Navbar horizontale avec état actif**
>
> Power BI ne gère pas l'état actif natif. Astuce :
>
> 1. Sur chaque page, créer 5 boutons **Texte** (un par section), tous avec action **Navigation de page** vers leur page respective
> 2. Sur la page courante, le bouton correspondant n'a PAS d'action (il pointe sur lui-même) et a un style différent : texte violet + soulignement violet 2px
> 3. Dupliquer la disposition des 5 boutons sur les 5 pages, en changeant à chaque fois le bouton actif

**Slicers globaux** : 2 slicers Date "Entre" `dim_calendrier[Date]` (`01/01/2023` à `31/12/2024`) en haut à droite. Clic droit → **Synchroniser les segments → cocher toutes les pages**.

**Logo « G GoogleAdsPulse »** : carré violet `#7C3AED` avec lettre G blanche en haut-gauche, suivi du nom et sous-titre « Paid Media Analytics » en gris.

---
# VIII — Validation et livraison

## 8.1 Checklist de recette

**Modèle** : 12 tables sources + dim_calendrier + _Mesures, 12 relations actives, 0 LocalDateTable · `dim_calendrier` marquée comme table de dates · Auto Date/Time désactivé.

**Mesures** : 82 dans `_Mesures` · 11 dossiers numérotés `0.` à `10.` + `_Helpers` · format défini (€, %, ×, K).

**Visuels HTML Content** : 1 visuel marketplace (Heatmap Jour × Heure) si tu choisis cette option.

**Pages** : 5 pages avec sous-titre dynamique · Slicers Date synchronisés · Navigation navbar horizontale avec état actif · Couleurs conformes à la charte Light + Violet.

## 8.2 Pièges fréquents

| Symptôme | Cause | Correction |
|---|---|---|
| `DATEADD(-1, MONTH)` renvoie blank | `dim_calendrier` non marquée comme table de dates | Outils de table → Marquer comme table de dates |
| Couleur delta CPC en rouge alors qu'il a baissé | Mesure `Couleur CPC` mal écrite (UP IS GOOD au lieu de DOWN IS GOOD) | Vérifier que la condition est `_d <= 0 → vert` (et pas `>= 0`) |
| Cohort retention affiche des totaux parasites | Totaux activés dans la matrice | Format → Totaux → Désactivé sur lignes et colonnes |
| ROAS Verdict = blanc/blank dans certaines cellules | Aucune valeur de ROAS pour cette campagne | Wrapper la mesure dans `IF(ISBLANK([ROAS]), "-", [ROAS Verdict])` |
| Top N filter ne respecte pas la mesure | Mauvaise mesure dans le « Par valeur » | Sélectionner `[ROAS]` (et pas une colonne du tableau) |
| Heatmap Jour × Heure : ordre des jours alphabétique | Power BI trie alphabétiquement par défaut | Trier `gads_jour_heure[jour]` par `jour_ordre` (1=Lun, 7=Dim) via Outils de colonne → Trier par colonne |
| Combo chart Impressions vs Clicks : axes mal échelonnés | Pas d'axe secondaire activé | Format → Axe Y → Activer Axe secondaire pour Total Clicks |
| Donut Conversions par type : pourcentages à 1 décimale différente | Format des étiquettes mal configuré | Format → Étiquettes de données → Format = Pourcentage 0 décimale |

## 8.3 Storytelling exécutif

Pour présenter à l'équipe média ou au client, suis l'ordre des 5 pages :

1. **Overview** : « Sur 24 mois, 5 comptes : 56 M d'impressions, 1,5 M de clics, 60K conversions, 3,7 M€ de revenu attribué pour 625K€ de spend → ROAS global 6×. Toutes les métriques en hausse vs M-1. »
2. **Campaigns** : « 25 campagnes actives. 5 campagnes Brand portent les 5 premiers ROAS (>11×). 5 campagnes en zone rouge (<2×) à couper ou retravailler — Search-Generic-Tech 2,6× est limite, Search-MedicalEquip 0,3× est à perte. »
3. **Keywords** : « 372 mots-clés, Quality Score moyen 6,7 (correct). 23 keywords à pauser (QS≤3 et spend>100€). Cohort retention stable autour de 95-100% sur tous les mois. »
4. **Conversions** : « 60K conversions. 63 % Purchase (Cost/Conv 11,3€), 17 % Lead (7,5€), 7 % Signup (13,3€). Top 3 campagnes par volume = PerfMax-LeadGen, Bookings, AllProducts. »
5. **Breakdown** : « Mobile 52 % du spend (€330K), Desktop 43 % (€265K), Tablet marginal. Pic de spend mardi-jeudi 9h-12h. Top pays = Côte d'Ivoire (€249K), Sénégal (€124K). Mix produits inchangé entre 2023 et 2024. »
6. **Recommandation** : couper les 5 campagnes en zone rouge (~50K€/an de spend récupéré) et réallouer sur PerfMax (ROAS 6-10×).

## 8.4 Annexes

### Mapping mockup PPTX ↔ pages Power BI

| Slide | Background PNG | Page |
|---|---|---|
| 1 | `bg-01-overview.png` | Overview |
| 2 | `bg-02-campaigns.png` | Campaigns |
| 3 | `bg-03-keywords.png` | Keywords |
| 4 | `bg-04-conversions.png` | Conversions |
| 5 | `bg-05-breakdown.png` | Breakdown |


---
<div style="background:#1E3A5F;padding:24px 32px;border-radius:10px;color:#FFFFFF;font-family:Georgia,serif;text-align:center;">
<div style="font-size:22px;font-weight:700;margin-bottom:6px;">GoogleAdsPulse — Paid Media Analytics</div>
<div style="font-size:13px;color:#CBD5E0;font-family:'Segoe UI',sans-serif;"><b>DataProjectLab</b> — apprendre la data sur des cas concrets, structurés et orientés métier.</div>
</div>